# Merge ATL, CPR and AC products — Antarctica / Ross Sea

Combines the three 1-second AN products into the K-means-ready datasets
`challenge_1min_numerical_AN.nc` and `challenge_1min_complete_AN.nc`.

**Inputs** (produced by the AN EDA notebooks):
`AC__TC__2B_1s_AN.nc`, `CPR_CLD_2A_1s_AN.nc`, `aot_resampled_an.nc`.

Adapted from `merger_EP.ipynb` — only the region suffix and the AOT path
differ. The spatial (Ross Sea / antimeridian) filtering already happened in
the EDA notebooks and is re-applied in `kmeans_AN.ipynb`.

In [ ]:
from pystac_client import Client
import fsspec
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import requests
from IPython.display import Image, display
import os
import pathlib

from scipy import stats  # will use scipy.stats.mode as the reducer
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

In [ ]:
# _suffix = '_WP'
# _suffix = '_EP'
_suffix = '_AN'

Load the synergetic classification (AC) and the cloud radar (CPR) products.

In [ ]:
a = xr.open_dataset(f"AC__TC__2B_1s{_suffix}.nc")
a

In [ ]:
a.time.plot(marker='o')

In [ ]:
b = xr.open_dataset(f"CPR_CLD_2A_1s{_suffix}.nc")
b

In [ ]:
b.time.plot(marker='o')

In [ ]:
challenge = a.merge(
    b[['land_flag', 'ice_water_path', 'liquid_water_path']],
    join='outer', compat='override')

Load the ATLID aerosol optical thickness produced by `atl_ald_2a_AN.ipynb`.

In [ ]:
# Local file produced by atl_ald_2a_AN.ipynb (copy it next to this notebook,
# or point to your bucket as in merger_EP.ipynb).
c = xr.open_dataset("aot_resampled_an.nc")
c

In [ ]:
c.time.plot(marker='o')

In [ ]:
challenge = challenge.merge(c, join='outer', compat='override')
challenge

In [ ]:
challenge.time.plot(marker='o')

## Fill clear-sky gaps with zero

When the sensors are working but the sky is clear, the retrieval is NaN;
under the clear-sky assumption we set those to 0 and clip rare artefacts.

In [ ]:
numerical_variables = [
    'latitude',
    'longitude',
    'land_flag',
    'ice_water_path',
    'liquid_water_path',
    'aerosol_optical_thickness_355nm',
]

In [ ]:
challenge[numerical_variables[2:]] = (
    challenge[numerical_variables[2:]].fillna(0).clip(min=0, max=1e2))

In [ ]:
challenge['latitude'].plot()

## Resample to 1-minute pixels

Numerical variables are averaged; classification labels are reduced by mode
(below) so aerosols and clouds share the same coarse pixel.

In [ ]:
challenge_1min_numerical = (
    challenge[numerical_variables]
    .resample(time='1min').mean()
    .dropna(dim='time', how='any'))
challenge_1min_numerical

In [ ]:
challenge_1min_numerical.plot.scatter(x='liquid_water_path', y='ice_water_path')

In [ ]:
challenge_1min_numerical.plot.scatter(x='aerosol_optical_thickness_355nm', y='ice_water_path')

## Save the K-means-ready numerical dataset

In [ ]:
challenge_1min_numerical.to_netcdf('challenge_1min_numerical_AN.nc')

## Resample the label variables by mode

In [ ]:
def mode_func(arr, axis):
    # scipy returns both mode and count, we only want the mode
    return stats.mode(arr, axis=axis, keepdims=False).mode

challenge_1min_labels = (
    challenge
    .drop_vars(numerical_variables)
    .resample(time="1min")
    .reduce(mode_func)
    .dropna(dim="time", how="any"))

challenge_1min_labels

In [ ]:
challenge_1min_complete = challenge_1min_numerical.merge(
    challenge_1min_labels, join='outer', compat='override')
challenge_1min_complete

## Save the complete dataset (numerical + TC labels)

In [ ]:
challenge_1min_complete.to_netcdf('challenge_1min_complete_AN.nc')

In [ ]:
%cp "challenge_1min_complete_AN.nc" "/home/jovyan/my-private-bucket/."